# GPS Flag Audit — 30 Participant Dataset

This notebook audits the **raw GPS observations** and identifies four types of quality flags:

1. Invalid coordinate range
2. Participant-specific coordinate IQR outliers
3. Speed diagnostic flags
4. Speed-accuracy diagnostic flags

### Important distinction

- **Coordinate flags** are used to identify GPS rows that are removed from the cleaned GPS dataset.
- **Speed and speed-accuracy flags are diagnostic only**. They are retained so we can inspect the quality of the movement measurements without automatically deleting otherwise valid latitude/longitude observations.

**Input file:** `unfiltered_30_participants_daily_features_with_gps.csv`

The notebook also creates participant-level summaries and PPT-ready charts.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Raw GPS input file
INPUT_FILE = "unfiltered_30_participants_daily_features_with_gps.csv"

df = pd.read_csv(INPUT_FILE)

print(f"Loaded {len(df):,} GPS rows")
print(f"Participants: {df['participant_id'].nunique()}")

display(df.head())


## 1. Basic data check

Before creating flags, confirm that the required GPS quality columns are available.


In [ ]:
required_columns = [
    "participant_id",
    "latitude",
    "longitude",
    "speed",
    "speed_accuracy"
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("All required columns are present.")
print()
print("Missing values:")
display(df[required_columns].isna().sum())


## 2. Coordinate range check

A coordinate is flagged if:

- latitude is outside -90 to +90
- longitude is outside -180 to +180
- latitude or longitude is exactly 0

These are treated as coordinate-quality problems.


In [ ]:
def flag_invalid_range(df):
    return (
        (df["latitude"] < -90) |
        (df["latitude"] > 90) |
        (df["longitude"] < -180) |
        (df["longitude"] > 180) |
        (df["latitude"] == 0) |
        (df["longitude"] == 0)
    )

df["flag_range"] = flag_invalid_range(df)

print(f"Invalid coordinate-range flags: {df['flag_range'].sum():,}")


## 3. Participant-specific IQR check

GPS locations are checked **within each participant**, rather than against one global distribution.

A point is flagged when latitude or longitude falls more than **3 × IQR** below Q1 or above Q3.

This prevents a location that is normal for one participant from being treated as an outlier simply because it is unusual for the whole cohort.


In [ ]:
IQR_MULTIPLIER = 3.0

def flag_iqr_outliers(group, multiplier=3.0):
    outlier = pd.Series(False, index=group.index)

    for col in ["latitude", "longitude"]:
        q1 = group[col].quantile(0.25)
        q3 = group[col].quantile(0.75)
        iqr = q3 - q1

        if pd.notna(iqr):
            outlier |= (
                (group[col] < q1 - multiplier * iqr) |
                (group[col] > q3 + multiplier * iqr)
            )

    return outlier

df["flag_iqr"] = (
    df.groupby("participant_id", group_keys=False)
      .apply(
          lambda g: flag_iqr_outliers(g, IQR_MULTIPLIER),
          include_groups=False
      )
      .reindex(df.index)
      .fillna(False)
      .astype(bool)
)

print(f"Coordinate IQR flags: {df['flag_iqr'].sum():,}")


## 4. Speed diagnostic flag

Speed is treated as a **diagnostic quality indicator**, not an automatic reason to remove the GPS coordinate.

The corrected analysis uses:

- `speed == -1` as the device's missing/unavailable sentinel
- `speed > 35 m/s` as the high-speed threshold


In [ ]:
MAX_SPEED_MS = 35.0

df["flag_speed"] = (
    (df["speed"] == -1) |
    (df["speed"] > MAX_SPEED_MS)
)

print(f"Speed diagnostic flags: {df['flag_speed'].sum():,}")


## 5. Speed-accuracy diagnostic flag

Speed accuracy is also retained as a diagnostic.

The corrected analysis uses:

- `speed_accuracy == -1` as the unavailable sentinel
- `speed_accuracy > 5 m/s` as the poor-accuracy threshold


In [ ]:
MAX_SPEED_ACCURACY = 5.0

df["flag_speed_accuracy"] = (
    (df["speed_accuracy"] == -1) |
    (df["speed_accuracy"] > MAX_SPEED_ACCURACY)
)

print(
    f"Speed-accuracy diagnostic flags: "
    f"{df['flag_speed_accuracy'].sum():,}"
)


## 6. Separate cleaning flags from diagnostic flags

This is the key part of the audit.

### Used for GPS-row removal
- `flag_range`
- `flag_iqr`

### Kept as diagnostics
- `flag_speed`
- `flag_speed_accuracy`


In [ ]:
location_flag_cols = ["flag_range", "flag_iqr"]
diagnostic_flag_cols = ["flag_speed", "flag_speed_accuracy"]

df["is_location_outlier"] = df[location_flag_cols].any(axis=1)

df["is_any_flag"] = df[
    location_flag_cols + diagnostic_flag_cols
].any(axis=1)

df["methods_flagged"] = df[
    location_flag_cols + diagnostic_flag_cols
].sum(axis=1)

df["is_overlap"] = df["methods_flagged"] >= 2

def flag_reason(row):
    reasons = []

    if row["flag_range"]:
        reasons.append("Invalid coordinate range")
    if row["flag_iqr"]:
        reasons.append("Participant-specific IQR")
    if row["flag_speed"]:
        reasons.append("Speed diagnostic")
    if row["flag_speed_accuracy"]:
        reasons.append("Speed-accuracy diagnostic")

    return " + ".join(reasons) if reasons else "No flag"

df["flag_reason"] = df.apply(flag_reason, axis=1)

print(f"Coordinate rows to remove: {df['is_location_outlier'].sum():,}")
print(f"Rows with any flag: {df['is_any_flag'].sum():,}")


## 7. Global flag summary

The counts below are **flags raised by each rule**.

They should not be added together because the same GPS row can receive multiple flags.


In [ ]:
flag_counts = {
    "Coordinate IQR": int(df["flag_iqr"].sum()),
    "Invalid range": int(df["flag_range"].sum()),
    "Speed flag": int(df["flag_speed"].sum()),
    "Speed accuracy": int(df["flag_speed_accuracy"].sum())
}

summary_table = pd.DataFrame(
    {
        "Flag type": list(flag_counts.keys()),
        "GPS rows flagged": list(flag_counts.values())
    }
)

display(summary_table)

print(f"Total raw GPS rows: {len(df):,}")
print(f"Coordinate rows removed: {df['is_location_outlier'].sum():,}")
print(f"Clean GPS rows retained: {(~df['is_location_outlier']).sum():,}")


## 8. PPT-ready chart — GPS quality flags

This chart shows the number of GPS rows flagged by each rule.

**How to explain it:** coordinate flags are used for cleaning, while speed and speed-accuracy flags are retained as diagnostic information.


In [ ]:
plt.figure(figsize=(10.5, 5.8))

bars = plt.bar(
    list(flag_counts.keys()),
    list(flag_counts.values())
)

plt.ylabel("GPS rows flagged")
plt.title(
    "GPS quality flags raised by each detection rule",
    loc="left",
    fontsize=16,
    fontweight="bold"
)

plt.grid(axis="y", alpha=0.25)
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.gca().spines["left"].set_visible(False)

for bar, value in zip(bars, flag_counts.values()):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + max(flag_counts.values()) * 0.02,
        f"{value:,}",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold"
    )

plt.figtext(
    0.5,
    0.01,
    "Counts are flags raised by each rule; one GPS row can receive multiple flags. "
    "Only coordinate flags are used for GPS-row removal.",
    ha="center",
    fontsize=9
)

plt.tight_layout(rect=[0, 0.06, 1, 1])

plt.savefig(
    "gps_quality_flags_for_ppt.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


## 9. Participant-level flag summary

This lets us see which of the 30 participants have:

- coordinate outliers
- speed flags
- speed-accuracy flags
- any flag at all



In [ ]:
participant_flag_summary = (
    df.groupby("participant_id")
      .agg(
          total_points=("participant_id", "size"),
          range_flags=("flag_range", "sum"),
          iqr_flags=("flag_iqr", "sum"),
          speed_flags=("flag_speed", "sum"),
          speed_accuracy_flags=("flag_speed_accuracy", "sum"),
          coordinate_removed=("is_location_outlier", "sum"),
          any_flag=("is_any_flag", "sum")
      )
      .reindex(sorted(df["participant_id"].unique()))
      .reset_index()
)

participant_flag_summary["coordinate_removed_pct"] = (
    participant_flag_summary["coordinate_removed"] /
    participant_flag_summary["total_points"] * 100
).round(2)

display(participant_flag_summary)

participant_flag_summary.to_csv(
    "gps_flag_summary_per_participant.csv",
    index=False
)


## 10. Inspect the exact flagged GPS rows

The next cells export the exact observations so that every flag can be audited.

This is especially useful when you want to demonstrate that a speed flag does **not necessarily mean the coordinate itself was removed**.


In [ ]:
flagged_columns = [
    "participant_id",
    "date",
    "time",
    "latitude",
    "longitude",
    "speed",
    "speed_accuracy",
    "flag_range",
    "flag_iqr",
    "flag_speed",
    "flag_speed_accuracy",
    "is_location_outlier",
    "flag_reason"
]

flagged_gps = df.loc[df["is_any_flag"], flagged_columns].copy()

flagged_gps.to_csv(
    "gps_flagged_points_for_audit.csv",
    index=False
)

print(f"Rows with at least one flag: {len(flagged_gps):,}")
display(flagged_gps.head(50))


### 10.1 Exact speed-flagged rows


In [ ]:
speed_flagged = df.loc[
    df["flag_speed"],
    flagged_columns
].copy()

speed_flagged.to_csv(
    "speed_flagged_gps_points.csv",
    index=False
)

print(f"Speed-flagged rows: {len(speed_flagged):,}")
display(speed_flagged.head(50))


### 10.2 Exact speed-accuracy-flagged rows


In [ ]:
speed_accuracy_flagged = df.loc[
    df["flag_speed_accuracy"],
    flagged_columns
].copy()

speed_accuracy_flagged.to_csv(
    "speed_accuracy_flagged_gps_points.csv",
    index=False
)

print(
    f"Speed-accuracy flagged rows: "
    f"{len(speed_accuracy_flagged):,}"
)
display(speed_accuracy_flagged.head(50))


## 11. Coordinate outliers actually removed

This file contains only the GPS rows that the corrected cleaning logic removes.

No latitude/longitude is replaced with `0` or with a cleaning-generated `NaN`.


In [ ]:
removed_gps = df.loc[df["is_location_outlier"]].copy()

removed_gps.to_csv(
    "gps_coordinate_outliers_removed.csv",
    index=False
)

clean_gps = df.loc[~df["is_location_outlier"]].copy()

clean_gps.to_csv(
    "gps_outliers_removed.csv",
    index=False
)

print(f"Removed GPS rows: {len(removed_gps):,}")
print(f"Clean GPS rows: {len(clean_gps):,}")

# Sanity checks
assert clean_gps["latitude"].notna().all()
assert clean_gps["longitude"].notna().all()
assert not (clean_gps["latitude"] == 0).any()
assert not (clean_gps["longitude"] == 0).any()

print("Clean GPS sanity checks passed.")


## 12. Final audit interpretation

Use this wording in the presentation:

> **“We used coordinate-based checks to decide which GPS observations should be removed. Speed and speed-accuracy flags were kept as diagnostics because a questionable speed measurement does not automatically mean that the latitude and longitude are incorrect.”**

The exact flagged rows are available in the exported audit CSV files.
